<a href="https://colab.research.google.com/github/teannacottey/Holiday-Planner-Project/blob/tavonga-branch/Flights_Data_Holiday_Planner_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Holiday Planner Project — Flights Data Collection & Aggregation

This notebook completes the **flights data** section of the Holiday Planner project.

The aim is to create two final datasets for the Streamlit app:

- `destinations_flights_agg.csv` — one row per destination
- `destinations_flights_agg_charts.csv` — one row per destination and month

The September data was collected using the Google Flights API / Apify output. October, November and December were estimated from the September baseline because the API became unreliable for later runs and the project had a limited scraping budget.


## 1. Import libraries

This follows the same simple structure used in the project app file: import the required libraries first, then import or create the datasets.


In [57]:
# import libraries
import pandas as pd
import numpy as np
from pathlib import Path

# show all columns when checking the data
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 2. Set file paths

In Google Colab, upload the source file into the notebook session or mount Google Drive.  
For the GitHub project, save the final outputs inside:

`data/flights/`


In [58]:
# set project paths
BASE_PATH = Path('.')
FLIGHTS_PATH = BASE_PATH / 'data' / 'flights'
FLIGHTS_PATH.mkdir(parents=True, exist_ok=True)

# source file used to build the final outputs
# in Colab, upload this file or change the path to your Drive location
RAW_SOURCE_FILE = 'destinations_flights_raw_full.csv'

print('Flights folder:', FLIGHTS_PATH)


Flights folder: data/flights


## 3. Load the flights dataset

The raw dataset contains:

- observed September API rows
- estimated October, November and December rows
- route details such as price, duration, stops and airline

This gives the app enough data to compare travel cost and travel time across the 12 holiday destinations.


In [59]:
flights_raw = pd.read_csv('destinations_flights_raw_full.csv')
flights_raw.head()


,destination,country,month,origin_airport,arrival_airport,arrival_airport_name,airport_used_type,departure_date,return_date,airline,price_gbp,duration_minutes,duration_hours,stops,stopover_airports,outbound_route,currency,google_search_url,data_source,estimation_method
0,Bali,Indonesia,Sep,LHR,DPS,Ngurah Rai International Airport,Exact,2026-09-01,2026-09-08,China Southern,785.0,1110,18.50,1,CAN,LHR->CAN; CAN->DPS,GBP,https://www.google.com/travel/flights?output=s...,Apify Google Flights API result,Observed September API output
1,Bali,Indonesia,Sep,LHR,DPS,Ngurah Rai International Airport,Exact,2026-09-01,2026-09-08,Etihad,809.0,1125,18.75,1,AUH,LHR->AUH; AUH->DPS,GBP,https://www.google.com/travel/flights?output=s...,Apify Google Flights API result,Observed September API output
2,Bali,Indonesia,Sep,LHR,DPS,Ngurah Rai International Airport,Exact,2026-09-01,2026-09-08,Singapore Airlines,1025.0,1235,20.58,1,SIN,LHR->SIN; SIN->DPS,GBP,https://www.google.com/travel/flights?output=s...,Apify Google Flights API result,Observed September API output
3,Bali,Indonesia,Oct,LHR,DPS,Ngurah Rai International Airport,Exact,2026-10-01,2026-10-08,Singapore Airlines,845.0,1180,19.67,1,SIN,LHR->SIN; SIN->DPS,GBP,https://www.google.com/travel/flights?output=s...,"Estimated from September API baseline, not liv...",September observed average price × 0.97; durat...
4,Bali,Indonesia,Nov,LHR,DPS,Ngurah Rai International Airport,Exact,2026-11-01,2026-11-08,Singapore Airlines,875.0,1192,19.86,1,SIN,LHR->SIN; SIN->DPS,GBP,https://www.google.com/travel/flights?output=s...,"Estimated from September API baseline, not liv...",September observed average price × 1.00; durat...


## 4. Check the data before cleaning

Before creating the final CSV files, check the shape, duplicates and missing values.  
This matches the project approach of validating the dataset before transformation.


In [60]:
# check rows and columns before cleaning
print('Rows and columns before cleaning:', flights_raw.shape)

# check duplicates
print('Duplicate rows before cleaning:', flights_raw.duplicated().sum())

# check missing values
flights_raw.isna().sum().sort_values(ascending=False)


Rows and columns before cleaning: (73, 20)
Duplicate rows before cleaning: 0


,0
stopover_airports,22
destination,0
month,0
country,0
origin_airport,0
arrival_airport,0
airport_used_type,0
arrival_airport_name,0
return_date,0
airline,0


## 5. Clean data types and remove duplicates

The cleaning is kept simple and replicable:

- remove duplicate rows
- standardise month order
- convert price, duration and stops to numeric values
- keep the source and estimation columns for transparency


In [61]:
# remove duplicate rows
flights_clean = flights_raw.drop_duplicates().copy()

# define the month order used in the app charts
month_order = ['Sep', 'Oct', 'Nov', 'Dec']

# clean month column
flights_clean['month'] = pd.Categorical(
    flights_clean['month'],
    categories=month_order,
    ordered=True
)

# convert numeric columns
numeric_columns = ['price_gbp', 'duration_minutes', 'duration_hours', 'stops']

for column in numeric_columns:
    flights_clean[column] = pd.to_numeric(flights_clean[column], errors='coerce')

# sort data for readability
flights_clean = flights_clean.sort_values(['destination', 'month']).reset_index(drop=True)

# check rows and columns after cleaning
print('Rows and columns after cleaning:', flights_clean.shape)
print('Duplicate rows after cleaning:', flights_clean.duplicated().sum())

flights_clean.head()


Rows and columns after cleaning: (73, 20)
Duplicate rows after cleaning: 0


,destination,country,month,origin_airport,arrival_airport,arrival_airport_name,airport_used_type,departure_date,return_date,airline,price_gbp,duration_minutes,duration_hours,stops,stopover_airports,outbound_route,currency,google_search_url,data_source,estimation_method
0,Bali,Indonesia,Sep,LHR,DPS,Ngurah Rai International Airport,Exact,2026-09-01,2026-09-08,China Southern,785.0,1110,18.50,1,CAN,LHR->CAN; CAN->DPS,GBP,https://www.google.com/travel/flights?output=s...,Apify Google Flights API result,Observed September API output
1,Bali,Indonesia,Sep,LHR,DPS,Ngurah Rai International Airport,Exact,2026-09-01,2026-09-08,Etihad,809.0,1125,18.75,1,AUH,LHR->AUH; AUH->DPS,GBP,https://www.google.com/travel/flights?output=s...,Apify Google Flights API result,Observed September API output
2,Bali,Indonesia,Sep,LHR,DPS,Ngurah Rai International Airport,Exact,2026-09-01,2026-09-08,Singapore Airlines,1025.0,1235,20.58,1,SIN,LHR->SIN; SIN->DPS,GBP,https://www.google.com/travel/flights?output=s...,Apify Google Flights API result,Observed September API output
3,Bali,Indonesia,Oct,LHR,DPS,Ngurah Rai International Airport,Exact,2026-10-01,2026-10-08,Singapore Airlines,845.0,1180,19.67,1,SIN,LHR->SIN; SIN->DPS,GBP,https://www.google.com/travel/flights?output=s...,"Estimated from September API baseline, not liv...",September observed average price × 0.97; durat...
4,Bali,Indonesia,Nov,LHR,DPS,Ngurah Rai International Airport,Exact,2026-11-01,2026-11-08,Singapore Airlines,875.0,1192,19.86,1,SIN,LHR->SIN; SIN->DPS,GBP,https://www.google.com/travel/flights?output=s...,"Estimated from September API baseline, not liv...",September observed average price × 1.00; durat...


## 6. Check destination coverage

The project uses 12 destinations:

Tenerife, Ibiza, Mykonos, Cancun, Bali, Bangkok, Maldives, Phuket, Holetown, Sal, Dubrovnik and Miami.


In [62]:
# expected destinations from the Streamlit app
expected_destinations = [
    'Tenerife', 'Ibiza', 'Mykonos', 'Cancun',
    'Bali', 'Bangkok', 'Maldives', 'Phuket',
    'Holetown', 'Sal', 'Dubrovnik', 'Miami'
]

# check which destinations are present
available_destinations = sorted(flights_clean['destination'].dropna().unique())
missing_destinations = sorted(set(expected_destinations) - set(available_destinations))

print('Available destinations:', available_destinations)
print('Missing destinations:', missing_destinations)

# check month coverage by destination
coverage_check = (
    flights_clean
    .groupby('destination', observed=False)['month']
    .nunique()
    .reset_index(name='months_available')
    .sort_values('destination')
)

coverage_check


Available destinations: ['Bali', 'Bangkok', 'Cancun', 'Dubrovnik', 'Holetown', 'Ibiza', 'Maldives', 'Miami', 'Mykonos', 'Phuket', 'Sal', 'Tenerife']
Missing destinations: []


,destination,months_available
0,Bali,4
1,Bangkok,4
2,Cancun,4
3,Dubrovnik,4
4,Holetown,4
5,Ibiza,4
6,Maldives,4
7,Miami,4
8,Mykonos,4
9,Phuket,4


## 7. Create destination-level aggregation

This output is used for overall recommendation scoring.  
It gives one row per destination with average price, cheapest price, average duration and average number of stops.

Output file:

`destinations_flights_agg.csv`


In [63]:
# aggregate by destination
flights_agg = (
    flights_clean
    .groupby(
        [
            'destination',
            'country',
            'origin_airport',
            'arrival_airport',
            'arrival_airport_name',
            'airport_used_type'
        ],
        as_index=False,
        observed=False
    )
    .agg(
        average_price_gbp=('price_gbp', 'mean'),
        cheapest_price_gbp=('price_gbp', 'min'),
        highest_price_gbp=('price_gbp', 'max'),
        average_duration_minutes=('duration_minutes', 'mean'),
        average_duration_hours=('duration_hours', 'mean'),
        average_stops=('stops', 'mean'),
        flight_options_count=('price_gbp', 'count'),
        estimated_rows_count=('data_source', lambda x: (x.astype(str).str.contains('Estimated', case=False)).sum())
    )
)

# round numeric values
rounding_rules = {
    'average_price_gbp': 2,
    'cheapest_price_gbp': 2,
    'highest_price_gbp': 2,
    'average_duration_minutes': 0,
    'average_duration_hours': 2,
    'average_stops': 2
}

flights_agg = flights_agg.round(rounding_rules)

# sort by average price so cheaper destinations are easier to see
flights_agg = flights_agg.sort_values('average_price_gbp').reset_index(drop=True)

flights_agg


,destination,country,origin_airport,arrival_airport,arrival_airport_name,airport_used_type,average_price_gbp,cheapest_price_gbp,highest_price_gbp,average_duration_minutes,average_duration_hours,average_stops,flight_options_count,estimated_rows_count
0,Dubrovnik,Croatia,LHR,DBV,Dubrovnik Ruđer Bošković Airport,Exact,198.43,170.0,245.0,280.0,4.67,0.86,7,3
1,Ibiza,Spain,LHR,IBZ,Ibiza Airport,Exact,214.29,180.0,322.0,315.0,5.24,0.86,7,3
2,Mykonos,Greece,LHR,JMK,"Mykonos National Airport ""Manto Mavrogeni""",Exact,274.67,240.0,297.0,278.0,4.64,0.33,6,3
3,Tenerife,Spain,LHR,TFS,Tenerife South Airport,Exact,279.50,211.0,326.0,624.0,10.39,1.00,6,3
4,Miami,United States,LHR,MIA,Miami International Airport,Exact,530.29,478.0,635.0,646.0,10.77,0.14,7,3
5,Bangkok,Thailand,LHR,BKK,Suvarnabhumi Airport,Exact,672.83,630.0,785.0,932.0,15.53,1.00,6,3
6,Holetown,Barbados,LHR,BGI,Grantley Adams International Airport,Exact,708.17,645.0,865.0,681.0,11.34,0.17,6,3
7,Phuket,Thailand,LHR,HKT,Phuket International Airport,Exact,727.50,660.0,855.0,1004.0,16.74,1.00,4,4
8,Cancun,Mexico,LHR,CUN,Cancún International Airport,Exact,762.29,695.0,920.0,916.0,15.26,1.00,7,3
9,Bali,Indonesia,LHR,DPS,Ngurah Rai International Airport,Exact,900.67,785.0,1065.0,1176.0,19.60,1.00,6,3


## 8. Create destination × month aggregation for charts

  It allows the app to show a line chart comparing flight prices and duration across September, October, November and December.

Output file:

`destinations_flights_agg_charts.csv`


In [64]:
# aggregate by destination and month
flights_clean['month'] = flights_clean['month'].astype(str).str.strip()

flights_agg_charts = (
    flights_clean
    .groupby(
        [
            'destination',
            'country',
            'month',
            'origin_airport',
            'arrival_airport',
            'arrival_airport_name'
        ],
        as_index=False,
        observed=True
    )
    .agg(
        average_price_gbp=('price_gbp', 'mean'),
        cheapest_price_gbp=('price_gbp', 'min'),
        highest_price_gbp=('price_gbp', 'max'),
        average_duration_minutes=('duration_minutes', 'mean'),
        average_duration_hours=('duration_hours', 'mean'),
        average_stops=('stops', 'mean'),
        flight_options_count=('price_gbp', 'count')
    )
)

# round values
flights_agg_charts = flights_agg_charts.round(rounding_rules)

# sort values for chart order
month_order = ['Sep', 'Oct', 'Nov', 'Dec']
month_rank = {month: position for position, month in enumerate(month_order, start=1)}

flights_agg_charts['month_order'] = flights_agg_charts['month'].map(month_rank)
flights_agg_charts = flights_agg_charts.sort_values(
    ['destination', 'month_order']
).reset_index(drop=True)

flights_agg_charts = flights_agg_charts.drop(columns=['month_order'])

flights_agg_charts.head(20)


,destination,country,month,origin_airport,arrival_airport,arrival_airport_name,average_price_gbp,cheapest_price_gbp,highest_price_gbp,average_duration_minutes,average_duration_hours,average_stops,flight_options_count
0,Bali,Indonesia,Sep,LHR,DPS,Ngurah Rai International Airport,873.00,785.0,1025.0,1157.0,19.28,1.00,3
1,Bali,Indonesia,Oct,LHR,DPS,Ngurah Rai International Airport,845.00,845.0,845.0,1180.0,19.67,1.00,1
2,Bali,Indonesia,Nov,LHR,DPS,Ngurah Rai International Airport,875.00,875.0,875.0,1192.0,19.86,1.00,1
3,Bali,Indonesia,Dec,LHR,DPS,Ngurah Rai International Airport,1065.00,1065.0,1065.0,1214.0,20.24,1.00,1
4,Bangkok,Thailand,Sep,LHR,BKK,Suvarnabhumi Airport,655.67,633.0,696.0,917.0,15.28,1.00,3
5,Bangkok,Thailand,Oct,LHR,BKK,Suvarnabhumi Airport,630.00,630.0,630.0,935.0,15.59,1.00,1
6,Bangkok,Thailand,Nov,LHR,BKK,Suvarnabhumi Airport,655.00,655.0,655.0,944.0,15.74,1.00,1
7,Bangkok,Thailand,Dec,LHR,BKK,Suvarnabhumi Airport,785.00,785.0,785.0,962.0,16.04,1.00,1
8,Cancun,Mexico,Sep,LHR,CUN,Cancún International Airport,741.50,714.0,768.0,903.0,15.05,1.00,4
9,Cancun,Mexico,Oct,LHR,CUN,Cancún International Airport,695.00,695.0,695.0,921.0,15.35,1.00,1


## 9. Export final CSV files

These are the final files that should be added to the GitHub repo under:

`data/flights/`


In [65]:
# export final CSV files
raw_output_path = FLIGHTS_PATH / 'destinations_flights_raw.csv'
agg_output_path = FLIGHTS_PATH / 'destinations_flights_agg.csv'
charts_output_path = FLIGHTS_PATH / 'destinations_flights_agg_charts.csv'

flights_clean.to_csv(raw_output_path, index=False)
flights_agg.to_csv(agg_output_path, index=False)
flights_agg_charts.to_csv(charts_output_path, index=False)

print('Saved files:')
print(raw_output_path)
print(agg_output_path)
print(charts_output_path)


Saved files:
data/flights/destinations_flights_raw.csv
data/flights/destinations_flights_agg.csv
data/flights/destinations_flights_agg_charts.csv


## 10. Final quality checks

These checks confirm that the files have been created properly and that the chart dataset has the required 12 destinations and 4 months.


In [66]:
# check final output files
print('Raw data shape:', flights_clean.shape)
print('Destination aggregation shape:', flights_agg.shape)
print('Chart aggregation shape:', flights_agg_charts.shape)

print('Destinations in chart file:', flights_agg_charts['destination'].nunique())
print('Months in chart file:', flights_agg_charts['month'].nunique())

# preview final chart file
flights_agg_charts.head(20)


Raw data shape: (73, 20)
Destination aggregation shape: (12, 14)
Chart aggregation shape: (48, 13)
Destinations in chart file: 12
Months in chart file: 4


,destination,country,month,origin_airport,arrival_airport,arrival_airport_name,average_price_gbp,cheapest_price_gbp,highest_price_gbp,average_duration_minutes,average_duration_hours,average_stops,flight_options_count
0,Bali,Indonesia,Sep,LHR,DPS,Ngurah Rai International Airport,873.00,785.0,1025.0,1157.0,19.28,1.00,3
1,Bali,Indonesia,Oct,LHR,DPS,Ngurah Rai International Airport,845.00,845.0,845.0,1180.0,19.67,1.00,1
2,Bali,Indonesia,Nov,LHR,DPS,Ngurah Rai International Airport,875.00,875.0,875.0,1192.0,19.86,1.00,1
3,Bali,Indonesia,Dec,LHR,DPS,Ngurah Rai International Airport,1065.00,1065.0,1065.0,1214.0,20.24,1.00,1
4,Bangkok,Thailand,Sep,LHR,BKK,Suvarnabhumi Airport,655.67,633.0,696.0,917.0,15.28,1.00,3
5,Bangkok,Thailand,Oct,LHR,BKK,Suvarnabhumi Airport,630.00,630.0,630.0,935.0,15.59,1.00,1
6,Bangkok,Thailand,Nov,LHR,BKK,Suvarnabhumi Airport,655.00,655.0,655.0,944.0,15.74,1.00,1
7,Bangkok,Thailand,Dec,LHR,BKK,Suvarnabhumi Airport,785.00,785.0,785.0,962.0,16.04,1.00,1
8,Cancun,Mexico,Sep,LHR,CUN,Cancún International Airport,741.50,714.0,768.0,903.0,15.05,1.00,4
9,Cancun,Mexico,Oct,LHR,CUN,Cancún International Airport,695.00,695.0,695.0,921.0,15.35,1.00,1


## 11. Quick visual check

This chart checks whether the destination × month file is working before it goes into Streamlit.


In [67]:
# quick visual check
import plotly.graph_objects as go

destination = 'Sal'
filtered = flights_agg_charts[flights_agg_charts['destination'] == destination].copy()

chart = go.Figure()

chart.add_trace(
    go.Scatter(
        x=filtered['month'],
        y=filtered['average_price_gbp'],
        name='Average Price (£)',
        mode='lines+markers'
    )
)

chart.add_trace(
    go.Scatter(
        x=filtered['month'],
        y=filtered['average_duration_hours'],
        name='Average Duration (Hours)',
        mode='lines+markers',
        yaxis='y2'
    )
)

chart.update_layout(
    title=f"{destination}'s End of Year Flight Prices and Duration",
    xaxis_title='Month',
    yaxis=dict(title='Average Price (£)'),
    yaxis2=dict(
        title='Average Duration (Hours)',
        overlaying='y',
        side='right'
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    hovermode='x unified'
)

chart.show()
